# Gram Newton-Schulz
## The Goal
Let $G \in \R^{m \times n}$ where $m \leq n = \alpha m$. We want to compute
$$\mathrm{polar}(G) = (GG^\top)^{-1/2} G$$

The standard Newton-Schulz-like iteration is $$X_0 = G, \quad X_t = p_t(X_{t-1})$$ where $p_t$ is a polynomial like $p_t(M) = \tfrac32 M - \tfrac12 MM^\top M$. Here, $(p_1, p_2, \ldots)$ is a sequence of odd polynomials satisfying $$\lim_{T \to \infty} (p_T \circ \cdots \circ p_1)(x) \to 1 \quad \forall x \in (0, 1]$$
This ensures that Newton Schulz converges for all $G$ such that $\|G\|_2 \leq 1$.
If run for $T$ iterations, the runtime of Newton Schulz is $\approx 2\alpha T m^3$.

The alternative version aims to do as much computation as possible with small $m \times m$ matrices. We call this method "Gram Newton Schulz". The outline of the method is as follows:
1. Form the Gram matrix $GG^\top$
2. Compute the approximation $Q_T \approx (GG^\top)^{-1/2}$ using an iterative polynomial method
3. Form $Q_T G$

Steps 1 and 3 each require multiplying $m \times n$ matrices. However, step 2 does not. Therefore, the runtime of Gram Newton Schulz is just $(3T + 2\alpha)m^3$. Figuring $T=5, \alpha=4$, this version is 43\% faster.

When we instead use degree-5 polynomials $p_t(x) = ax + bx^3 + cx^5$, standard Newton Schulz costs $(2\alpha + 1)Tm^3$ while Gram Newton Schulz costs $(4T + 2\alpha)m^3$, 38\% faster.
Additionally, we can use symmetric matrix multiplication routines which perform multiplications like $XX^\top$ by computing the upper triangular half of the output and then copying the results to the lower triangular half.
Using these kernels, standard Newton Schulz with degree-5 polynomials costs $\tfrac12(3\alpha + 1)Tm^3$ and our version costs $\tfrac12(4T + 3\alpha - 3)m^3$.
Again figuring $T=5, \alpha=4$, our version is 55% faster.

## Derivation from Standard Newton Schulz
It remains to construct an approximation to $(GG^\top)^{-1/2}$. As wtih Newton Schulz, we restrict ourselves to methods based on matrix polynomials.
As it turns out, such a method can be derived from any sequence of odd polynomials $(p_1, p_2, \ldots)$ that satisfies the convergence criterion for Newton Schulz given above. If we use this 
If we use this method in step 2, then the overall Gram Newton Schulz procedure will be exactly equivalent to standard Newton Schulz in exact arithmetic.

We begin with the scalar analogue of the method.
Let $x_0 \in (0, 1]$ and $x_t = p_t(x_{t-1})$, as in standard Newton Schulz.
Since $p_t$ is odd, it can be written as $p_t(x) = x h_t(x^2)$, where $h_t$ is a polynomial like $h_t(x) = \tfrac32 - \tfrac12 x$.
Define 
Define $r_t := x_t^2$ and $z_t := h_t(r_{t-1})$. Then
$$x_{t} = p_t(x_{t-1}) = x_{t-1} h_t(x_{t-1}^2) = x_{t-1} h_t(r_{t-1}) = x_{t-1} z_t$$
Squaring both sides yields
$$r_t = r_{t-1} z_t^2$$
If we define $q_t := x_t / x_0$, then we can also derive
$$q_t = q_{t-1} z_t$$
Finally, $1 = \lim_{T \to \infty} x_T = \lim_{T \to \infty} q_T x_0 = \lim_{T \to \infty} q_T \sqrt{r_0} \implies q_T \to 1/\sqrt{r_0}$. Thus, the following iterative method computes the inverse square root of $r_0$:
- $z_t = h_t(r_{t-1})$
- $r_t = r_{t-1} z_t^2$
- $q_t = q_{t-1} z_t$&emsp;(where we initialize $q_0 = 1$).

Furthermore, if we initialize this method with $r_0 = x_0^2$, then our analysis shows that $x_T = q_T x_0$ is precisely the output of standard Newton Schulz. Thus, we have computed $x_T$ from $x_0$ without ever constructing the intermediate values $x_1, \ldots x_{T-1}$.

To obtain Gram Newton Schulz, we simply lift the above procedure to matrices.
As in standard Newton Schulz, each operation preserves eigenvectors/singular vectors.
Therefore, each eigenvalue / singular value evolves independently of the others according to the scalar iteration described above.

> ### Gram Newton Schulz (Version 1 - High Precision)
> Input: $G \in \R^{m \times n}$ with $m \leq n$ and $\|G\|_2 \leq 1$.
>
> Initialize: $R_0 = GG^\top$ and $Q_0 = I$.
>
> Repeat for $t=1, \ldots,\,T$:
> - $Z_t = h_t(R_{t-1})$&emsp;&emsp;(e.g. $\tfrac32 I - \tfrac12 R_{t-1}$)
> - $R_t = Z_t^\top R_{t-1} Z_t$
> - $Q_t = Q_{t-1}Z_t$
>
> Output: $X_T = Q_T G$.

This method is similar to that of Polar Express, Appendix F.

## Tracking the Spectra in Gram Newton Schulz 

We will now demonstrate Gram Newton Schulz on a simple synthetic example of a $128 \times 512$ matrix with an exponentially decaying spectrum. We will use the degree-5 Newton Schulz polynomial $p_t(x) = \tfrac{15}8 x - \tfrac{10}8 x^3 + \tfrac38 x^5$.

By construction, $Z_t, R_t$, and $Q_t$ are symmetric.
Because the only operations we perform are matrix polynomials, the eigenvectors of all of these matrices are identical, and they match the left singular vectors of $X_t$.
We can therefore plot the eigenvalues of $R_t$ and $Q_t$ against the corresponding singular values of $X_0$ to trace how each evolves according to the polynomial iterations.
Even though our method does not need to compute the intermediate matrices $X_1, \ldots X_{T-1}$, we do so here for demonstration.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.scale import AsinhScale
import numpy as np
import pandas as pd
import torch

%load_ext autoreload
%autoreload 2
from appF_diagnostic import *

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

plt.rcParams.update({
    # "text.usetex": True,
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "axes.unicode_minus": False,
    "font.size": 14,  # baseline text size
    "figure.constrained_layout.use": True
})

In [ ]:
n = 256
aspect_ratio = 4
spectrum = torch.logspace(-0.01, -6, steps=n, dtype=torch.float64, device=DEVICE)
G = spectrum2matrix(spectrum, aspect_ratio)

_, f64_diagnostics = PolarExpressDiagnostic(
    coeffs_name="ns5",
    steps=10,
    ambient_dtype=torch.float64,
)(G)

f64_diagnostics_html = spectrum_evolution_plot(f64_diagnostics).to_jshtml()
with open("html_plots/f64_diagnostics.html", "w") as f:
    f.write(f64_diagnostics_html)
HTML(f64_diagnostics_html)

Initially, we have $r_0 = x_0^2$, and $q_0 = 1$. As training procedes, $q_t$ approaches $1/\sqrt{r_0} = 1/x_0$, and so $x_t = q_t x_0$ approaches $1$. Since $r_t = x_t^2$, it too approaches 1. (We can also see the gap between $r_t$ and $1$ as a measure of the error in the approximation $q_t \approx 1/\sqrt{r_0}$, since $r_t = q_t^2 r_0$.) Note that if $x_0$ is close to 1, the method converges quickly, while if $x_0$ is close to zero, it converges slowly. After 10 iterations, the spectrum of $X_t$ is visually indistinguishable from $1$.

So far, this is all expected. As we showed above, Gram Newton Schulz is exactly equivalent to Newton Schulz in exact arithmetic.

## Trouble in Low Precision
Unfortunately, in floating point arithemtic, Gram Newton Schulz is not equivalent to the standard version; it is numerically unstable. Let's see what happens when we repeat the previous experiment in `bfloat16` arithmetic.

In [ ]:
_, f16_diagnostics = PolarExpressDiagnostic(
    coeffs_name="ns5",
    steps=10,
    ambient_dtype=torch.bfloat16
)(G)

f16_diagnostics_html = spectrum_evolution_plot(f16_diagnostics).to_jshtml()
with open("html_plots/f16_diagnostics.html", "w") as f:
    f.write(f16_diagnostics_html)
HTML(f16_diagnostics_html)

The first few iterations proceed as before. However, by step 7, we see unexpected behavior in the spectrum of $X$. The singular values that began near $0$ suddenly jump up above 1, instead of converging to 1 from below. By step 8, the algorithm is returning complete junk. What happened?

### Spurious Negative Eigenvalues
If you look closely, you can see that the trouble begins in the matrix $R$. By construction, $r_t = x_t^2 \geq 0$, so in exact arithmetic, $R_t$ should be a positive semidefinite matrix.
However, when using `bfloat16`, our plots show that $R$ has negative eigenvalues!
Because $G$ is numerically low rank, $R_0 = GG^\top$ has eigenvalues that are *numerically* equal to zero, and in `bfloat16`, a number like $-10^{-5}$ is numerically equal to zero.
Let's transform the y-axis to emphasize values close to zero and replot this.

In [ ]:
f16_diagnostics_zoomed_html = spectrum_evolution_plot(f16_diagnostics, yscale='asinh', yscale_kw=dict(linear_width=1e-4)).to_jshtml()
with open("html_plots/f16_diagnostics_zoomed.html", "w") as f:
    f.write(f16_diagnostics_zoomed_html)
HTML(f16_diagnostics_zoomed_html)

Now we see that from the very beginning, $R$ has tiny negative eigenvalues. These eigenvalues represent nothing about the original problem, they are just an artifact of floating point arithemtic. Therefore, we call them "spurious eigenvalues".

These spurious negative eigenvalues start small, but the plot shows that their magnitude grows quickly.
Let's understand mathematically why this happens. Recall the update rule:
$$r_t = r_{t-1} z_t^2 = r_{t-1} h_t(r_{t-1})^2$$
If we now substitute $h_t(x) = \tfrac{15}8 - \tfrac{10}8 x + \tfrac38 x^2$, and plot this update rule, we can see the problem:

In [ ]:
def h(x): return (15/8) - (10/8) * x + (3/8) * x**2
def next_r(r): return r * (h(r)**2)
xxx = np.linspace(-.2, 1, 1000)
fig, ax = plt.subplots()
ax.plot(xxx, next_r(xxx), label='$r h(r)^2$')
# ax.plot(xxx, xxx, '--', color='gray', linewidth=0.5, label='r')
ax.plot(xxx[:420], ((15/8)**2)*xxx[:420], '--', color='gray', linewidth=1, label=r"$\left(\frac{15}{8}\right)^2 r$")
ax.set_xlabel('$r$')
ax.legend(loc='lower right')

ax.spines['left'].set_position('zero')
ax.spines['bottom'].set_position('zero')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.savefig("html_plots/r_update_map.svg", format="svg", bbox_inches="tight")

As the plot shows, $r_t < \left(\tfrac{15}{8}\right)^2 r_{t-1}$. Thus, if $r_0 < 0$, the magnitude of the spurious eigenvalues grows exponentially! This sets off a chain reaction. As $r_t \to -\infty$, we get $z_t \to \infty$. This causes $q_t \to \infty$ and therefore also $x_t \to \infty$.
This problem cannot be fixed by choosing different polynomials. Conceptually, we are attempting to compute the inverse square root of a negative number. It cannot help but diverge.

To show that the spurious negative eigenvalues of $R_0$ are enough to cause this catestrophic failure, let's rerun the method with every operation in `float64` precision, except that we will convert $R_0$ from `float64` to `bfloat16` and then back to `float64` to induce a little floating point error. As you can see, even this causes a blowup.

In [ ]:
_, posthoc_f16_diagnostics = PolarExpressDiagnostic(
    coeffs_name="ns5",
    steps=10,
    ambient_dtype=torch.float64,
    xxt_posthoc_dtype=torch.bfloat16,
)(G)

posthoc_f16_diagnostics_html = spectrum_evolution_plot(posthoc_f16_diagnostics).to_jshtml()
with open("html_plots/posthoc_f16_diagnostics.html", "w") as f:
    f.write(posthoc_f16_diagnostics_html)
HTML(posthoc_f16_diagnostics_html)

## Eigenvector Drift
Spurious negative eigenvalues are not the only source of instability.
If we take as input a matrix that lacks small singular values (i.e., all $\geq 0.017$), then we do not observe any negative eigenvalues in the $R_t$, but we still see a moderate blow up in $X_t$.

In [ ]:
easy_spectrum = torch.logspace(-0.01, -1.75, steps=n, dtype=torch.float64, device=DEVICE)


easy_spectrum_PE = PolarExpressDiagnostic(
    coeffs_name="ns5",
    steps=10,
    # restarts=[2],  # Observe that restarting fixes this problem!
    ambient_dtype=torch.bfloat16,
    do_diagnostics=True,
)

_, easy_spectrum_diagnostics = easy_spectrum_PE(spectrum2matrix(easy_spectrum, aspect_ratio))
easy_spectrum_exact = easy_spectrum_PE.track_eigvals(easy_spectrum, r_shift=0)

HTML(spectrum_evolution_plot(easy_spectrum_diagnostics).to_jshtml())

The culprit seems to be eigenvector drift.
In exact arithmetic, the eigenvectors of all intermediate matrices match $ U$, the left singular vectors of $ X_0$, but in finite precision they do not.
This effect can be measured by observing how far $ U^\top  R_t  U$, $ U^\top  Q_t  U$, and $ U^\top  X_t  U$ are from being diagonal matrices.
The plot below shows that after several iterations, the eigenvectors of $ Q_t$ and $ X_t$ have drifted significantly.
At the same time, we see the eigen*values* of $ Q_t$ (and by extension, those of $ X_t$) diverge from where they should be in exact arithmetic.
The growing eigenvalues of $ Q_t$ seem to spill into one another.
The strength of this effect is less consistent than that of negative eigenvalues, but it is still harmful.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 4))

diag_axis = axes[0]
for m in ("R", "Q", "X"):
    diag_axis.plot(easy_spectrum_diagnostics[f"{m}_diagonalizability"], label=f"${m}_t$", marker='o')
diag_axis.legend()
diag_axis.set_xlabel("Step ($t$)")
diag_axis.set_ylabel("Relative\nDiagonalization Error")
diag_axis.set_ylim(top=max(0.2, diag_axis.get_ylim()[1]))

for ax, m, c in zip(axes[1:], ("R", "Q", "X"), ("C0", "C1", "C2")):
    # Frob norm: ax.plot([np.linalg.norm(spectrum) for spectrum in easy_spectrum_diagnostics['X_singvals']], label="Observed", marker='o', color='purple')
    ax.plot(easy_spectrum_diagnostics[f"{m}_max_singval"], label="Observed", marker='o', color=c)
    if m == "X":
        ax.axhline(1, label="Theoretical", color='black', linestyle='--')
        theoretical_max = 1
    else:
        ax.plot([spectrum.max() for spectrum in easy_spectrum_exact[m]], label="Theoretical", color='black', linestyle='--')
        theoretical_max = max(spectrum.max() for spectrum in easy_spectrum_exact[m])
    ax.legend()
    ax.set_xlabel("Step ($t$)")
    ax.set_ylabel(f"Max Eigenvalue")
    ax.set_title(f"${m}_t$")
    ax.set_ylim(top=min(theoretical_max * 5, 1.5 * ax.get_ylim()[1]))

plt.tight_layout()

# fig.savefig("html_plots/easy_spectrum_diagnostics.svg", format="svg", bbox_inches="tight")
# fig.savefig("html_plots/easy_spectrum_diagnostics_restart.svg", format="svg", bbox_inches="tight")

## Controlling Negative Eigenvalues by Restarting
If we run Gram Newton Schulz for more than a few iterations, the spurious negative eigenvalues grow unmanageably large. Our solution is simple: run Gram Newton Schulz for only a few iterations.
Rather than using Gram Newton Schulz to compute $X_T$ directly, we use it to compute, say, $X_5$ in a stable manner.
While $X_5$ is not a good approximation to $\lim_{T \to \infty} X_T = \mathrm{polar}(X_0)$, we are closer than when we started.
Now we can apply Gram Newton Schulz a second time on the input $X_5$ to compute $X_{10}$ stably.
We can repeat this over and over to reach whatever $T$ we like.
This restarting technique sacrifices some of the performance gains of Gram Newton Schulz, but it still offers a significant speedup over standard Newton Schulz.

> ### Gram Newton Schulz (Version 2 - Restart Every 5)
> Input: $G \in \R^{m \times n}$ with $m \leq n$ and $\|G\|_2 \leq 1$.
>
> Initialize $X_0 = G$
>
> Repeat for $t = 0, 5, 10, \ldots, T$
> - "Reinitialize" $R_t \gets X_tX_t^\top$ and $Q_t \gets I$.
>
> - Repeat for $s=t+1, \ldots, t+5$:
>   - $Z_s = h_s(R_{s-1})$&emsp;&emsp;(e.g. $\tfrac32 I - \tfrac12 R_{s-1}$)
>   - $R_s = Z_s^\top R_{s-1} Z_s$
>   - $Q_s = Q_{s-1}Z_s$
>
> - $X_{t+5} = Q_{t+5} X_t$
>
> Output: $X_T$

Below we plot this method. As before, we compute $X_{t+s} = Q_{t+s} X_t$ for diagnostic purposes, even though the algorithm does so only for $s=5$. 
As you can see, at iteration $5, 10, 20, 25$, and $30$, $Q$ resets to the identity. Therefore, the eigenvalues of $Q$ never grow beyond $\approx 12$.
Looking closely, you can also see that $R$ develops a some negative eigenvalues, but unlike before, the growth of these eigenvalues is controlled.
Each time we restart, we re-initialize $R = XX^\top$, eliminating any negative eigenvalues of large magnitude.

In [ ]:
_, restart5_diagnostics = PolarExpressDiagnostic(
    coeffs_name="ns5",
    steps=30,
    restarts=[5, 10, 15, 20, 25],
    ambient_dtype=torch.bfloat16
)(G)

frames = list(range(11)) + [15, 20, 25, 30]
restart5_diagnostics_html = spectrum_evolution_plot(restart5_diagnostics, frames=frames).to_jshtml()
with open("html_plots/restart5_diagnostics.html", "w") as f:
    f.write(restart5_diagnostics_html)
HTML(restart5_diagnostics_html)

# When to Restart
Is it safe to run for $r$ iterations without restarting? To avoid numerical trouble, we need to control the magnitude of $Q_r$, even when $R_0$ has spurious negative eigenvalues. (Because each $q_r \geq 1$, this is equivalent to controlling the condition number of $Q_r$.) The growth of $Q_r$ in turn depends on the specific sequence of polynomials we use.
Furthermore, since the polynomial $p_t$ changes at each iteration, it may not be ideal to restart at regular intervals.
Instead, we can choose when to restart adaptively, depending on the specific sequence of polynomials we have applied since the previous restart.

For the application to Muon, let's now switch over to five iterations of the PolarExpress polynomials.
To obtain a good balance of stability and speed, let's limit ourselves to a single restart.
When should this restart take place?
Using the scalar analogue of the method, let's simulate how the eigenvalues of $Q_t$ evolve when $R_0$ has eigenvalues in the range $[-10^{-4}, 1]$.
The plot below shows that restarting after 2 iterations ensures that $\|Q_t\|_2 \leq 30$ for all iterations. (Restarting after 3 gives a worse bound, but only slightly.)

In [ ]:
planted_negative_eigenvalue_size = 1e-4
polar1restart_results = {
    restart: PolarExpressDiagnostic(
        coeffs_name="polar5",
        steps=5,
        restarts=[restart],  # list(range(5)),
        ambient_dtype=torch.bfloat16,
        do_diagnostics=True,
    ).track_eigvals(torch.logspace(0, -10, steps=2048, device=DEVICE, dtype=torch.float64), r_shift=planted_negative_eigenvalue_size)
    for restart in [1, 2, 3, 4]
}

fig, ax = plt.subplots()

for restart, exact_eigvals in polar1restart_results.items():
    ax.plot([np.abs(m).max() for m in exact_eigvals["Q"]], label=f"{restart} step", marker='o')
ax.set_title('Max Eigenvalue of $Q_t$')
ax.set_xlabel('Step ($t$)')
ax.legend(title="Restart after")
ax.set_yscale('log')
# ax.set_ylim(.9, 100)

our_lim = max(np.abs(vals).max() for vals in polar1restart_results[2]["Q"])
ax.axhline(our_lim, color='black', linestyle='--', linewidth=0.5)
print(f"Max eigenvalue: {our_lim:.2f}")


Now let's run the full method with a restart after the second iteration on our test matrix.
Now it converges!

In [ ]:
_, final_diagnostics = PolarExpressDiagnostic(
    coeffs_name="polar5",
    steps=5,
    restarts=[2],
    ambient_dtype=torch.float16,
    do_diagnostics=True,
)(G)

HTML(spectrum_evolution_plot(final_diagnostics).to_jshtml())

Note that we use `float16` arithmetic, not `bfloat16`.
For our purposes, the higher precision of `float16` is more valuable than the larger range of `bfloat16` because the magnitude of our matrices is controlled to lie near 1.

### Further caution
While restarting greatly improves stability, the usual numerical precautions for Newton-Schulz must still be taken.
In rare cases, restarting can actually worsen them.
Like most other choices of Newton-Schulz coefficients, the Polar Express polynomials are designed to converge only when $\| X_0\| \leq 1$; any singular values larger than $1$ will diverge rapidly.
Even with a properly normalized input, singular values slightly greater than $1$ can develop due to numerical error.
This problem affects standard Newton-Schulz as well, so the Polar Express polynomials are typically adjusted according to the formula
$\tilde p_t(x) = p_t(x / 1.01)$.
This ensures that convergence even for singular values as large as $1.01$.

Below, we consider an input matrix whose largest singular value lies closer to 1 than before.
To elicit instability, we ignore our own advice and use `bfloat16`.
Naive Gram Newton-Schulz without restarting is stable for this input, as the input matrix is well-conditioned.
But if we restart after two iterations, it blows up.
This blow up occurs even if we stop using `bfloat16` 

In [ ]:
close_spectrum = torch.linspace(.99, 0.1, 1000, device=DEVICE, dtype=torch.float64)

close_start_PE = PolarExpressDiagnostic(
    coeffs_name="rescaled_polar5",
    steps=5,
    restarts=[2],
    ambient_dtype=torch.bfloat16,
    post_restart_ambient_dtype=torch.float64,
    do_diagnostics=True,
)

_, bad_conditioning_b16_diagnostics = close_start_PE(spectrum2matrix(close_spectrum, aspect_ratio))
close_spectrum_results = close_start_PE.track_eigvals(close_spectrum, r_shift=0)

HTML(spectrum_evolution_plot(bad_conditioning_b16_diagnostics).to_jshtml())

The following figure shows that at iteration 2, just after the restart has taken place, the largest eigenvalue of $R_t$ is larger than 1.
(Observe that in this example, $ Q_t$ behaves as expected.)
In fact, it is larger than 1.01.
Therefore, even the adjusted Polar Express polynomials cause a blow up.

In [ ]:
print(bad_conditioning_b16_diagnostics[['R_min_singval', 'R_max_singval']])

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 4))

diag_axis = axes[0]
for m in ("R", "Q", "X"):
    diag_axis.plot(bad_conditioning_b16_diagnostics[f"{m}_diagonalizability"], label=f"${m}_t$", marker='o')
diag_axis.legend()
diag_axis.set_xlabel("Step ($t$)")
diag_axis.set_ylabel("Relative\nDiagonalization Error")
diag_axis.set_ylim(top=max(0.2, diag_axis.get_ylim()[1]))

for ax, m, c in zip(axes[1:], ("R", "Q", "X"), ("C0", "C1", "C2")):
    ax.plot(bad_conditioning_b16_diagnostics[f"{m}_max_singval"][:-1], label="Observed", marker='o', color=c)
    if m == "X":
        ax.axhline(1, label="Theoretical", color='black', linestyle='--')
        theoretical_max = 1
    else:
        ax.plot([spectrum.max() for spectrum in close_spectrum_results[m]], label="Theoretical", color='black', linestyle='--')
        theoretical_max = max(spectrum.max() for spectrum in close_spectrum_results[m])
    ax.legend()
    ax.set_xlabel("Step ($t$)")
    ax.set_ylabel(f"Max Eigenvalue")
    ax.set_title(f"${m}_t$")
    ax.set_ylim(top=min(theoretical_max * 5, 1.5 * ax.get_ylim()[1]))

plt.tight_layout()

The solution is this case is simple. Use `float16`, or adjust the polynomials by a larger factor, such as 1.05.

# Conclusion
The cost savings of Gram Newton Schulz make it very attractive.
While this method is fundamentally more unstable than standard Newton Schulz, it can be coaxed into behaving with the proper care.
The understanding gleaned from these experiments gives us the confidence to use Gram Newton Schulz in practice.
However, users should be willing to monitor the method, and if they find instability, to adjust the hyperparameters (like changing $1.01 \to 1.05$ above).
If running more than five iterations with Polar Express polynomials, a second restart may be required.

A technique not explored here is to control spurious negative eigenvalues in $\mathbf X \mathbf X^\top$ by adding a small multiple of the identity matrix like $\epsilon I$.
This causes the eigenvalues $x$ to converge to $x / \sqrt{x^2 + \epsilon}$ instead of $1$, but this can be fixed by restarting (and *not* adding $\epsilon I$ after the final restart.)
This technique is more relevant to cases when better accuracy (and therefore a larger number of iterations) is required.

In our applications, we do not need very high accuracy.
When high accuracy is desired, the usual warnings about forming the Gram matrix apply.
Since forming the Gram matrix immediately squares the condition number, Gram Newton Schulz may not be appropriate in these cases.